# Week 2 — Describing data

Week 1 gave you a centre and a spread. This week adds the two things that make
a description usable: **shape**, and a picture.

It also answers the question week 1 left open. You have a sample. Everything you
compute from it — a mean, a standard deviation — is a *statistic*, and it would
have come out differently with a different sample. How much differently, and how
much should that worry you?

In [ ]:
import math9102 as m9
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

m9.use_house_style()

---

## Part 1 — From a sample to a population

Suppose you want the mean length of all the fish in a lake.

- That number is a **parameter**: a fixed fact about the population.
- It is almost certainly unknowable — you cannot catch every fish, and the
  population changes while you try.
- So you catch a sample and compute its mean. That is a **statistic**.

Two people fishing the same lake get different fish, so they get different
means. That is **sampling variation**, and it is not a mistake by either of
them.

### The sampling distribution

Imagine repeating the sampling many times and collecting all the means. Those
means have a distribution of their own — the **sampling distribution of the
mean**.

We can simulate exactly that.

In [ ]:
rng = np.random.default_rng(0)
population = rng.normal(10, 3, 200_000)

def sample_means(n, reps=4000):
    return rng.choice(population, size=(reps, n)).mean(axis=1)

for n in (6, 25, 100):
    means = sample_means(n)
    print(f"samples of {n:3d}: mean of means = {means.mean():.3f}, "
          f"SD of means = {means.std(ddof=1):.3f}")

Two things to read off:

1. **The mean of the sample means lands on the population mean.** The sample
   mean is an unbiased estimate — on average, it is right.
2. **The spread of the sample means shrinks as the sample grows.** That spread
   is the thing we care about, and it has a name.

### Standard error

The **standard error of the mean** is the standard deviation of the sampling
distribution. It measures how much a sample mean is likely to move about.

You do not have to simulate it. It has a formula:

In [ ]:
for n in (6, 25, 100):
    means = sample_means(n)
    print(f"n = {n:3d}:  simulated SD of means = {means.std(ddof=1):.3f},  "
          f"formula = {population.std(ddof=0) / np.sqrt(n):.3f}")

Note the **square root**. Quadrupling the sample halves the standard error, it
does not quarter it. Precision gets more expensive the more of it you buy.

In practice you have one sample, not thousands, so you estimate the standard
error from it:

In [ ]:
festival = m9.load_festival()
day1 = festival.day1.dropna()

# The corrected copy: the recorded data hold an entry error, which Part 4 finds.
clean = m9.load_festival(with_outlier=False).day1.dropna()

se = clean.std(ddof=1) / np.sqrt(len(clean))
print(f"standard error of the mean = {se:.4f}")

**Do not confuse the two.** The standard deviation describes how spread out the
*observations* are. The standard error describes how precisely you have pinned
down the *mean*. A large dataset of very variable things has a large SD and a
small SE.

### The Central Limit Theorem

The simulation above used a normal population, so it is unsurprising that the
means came out normal. The Central Limit Theorem says something much stronger.

> **For a sufficiently large sample size, the sampling distribution of the mean
> is approximately normal — whatever the shape of the population.**

Read that sentence twice, because it is very easy to remember it wrong. Two
things it does **not** say:

- It does **not** say your data become normal. The population is whatever it is.
- It does **not** say a fixed sample size is enough for every population. How
  large "sufficiently large" has to be depends on how far from normal the
  population is.

Here is all three of those claims, on a strongly skewed population:

In [ ]:
skewed = rng.exponential(1.0, 400_000)

print(f"population skew: {stats.skew(skewed):.2f}")
for n in (2, 10, 50, 200):
    means = rng.choice(skewed, size=(6000, n)).mean(axis=1)
    print(f"  mean of {n:3d}: skew of the sampling distribution = {stats.skew(means):.2f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 2.6))
axes[0].hist(skewed, bins=60, range=(0, 6))
axes[0].set_title("population")
for ax, n in zip(axes[1:], (2, 10, 50)):
    ax.hist(rng.choice(skewed, size=(6000, n)).mean(axis=1), bins=50)
    ax.set_title(f"mean of {n}")
for ax in axes:
    ax.set_yticks([])
fig.tight_layout();

The population stays exactly as skewed as it always was. The **sampling
distribution** straightens out as the sample grows — and at a sample of two it
has barely started.

**Why we care.** Almost every test in this module compares a statistic to a
distribution and asks how surprising it is. That reasoning needs the sampling
distribution's shape. The CLT is what lets us assume it.

---

## Part 2 — The normal distribution

In [ ]:
x = np.linspace(-4, 4, 400)
plt.plot(x, stats.norm.pdf(x))
plt.xlabel("standard deviations from the mean")
plt.ylabel("density");

Its properties:

- Symmetric about the mean, and bell-shaped.
- The mean, median and mode are the same value.
- It is completely specified by two numbers: the mean and the standard
  deviation. Nothing else about it can vary.
- It approaches the horizontal axis but never touches it.
- The total area under it is 1, and **area is probability**.

### The empirical rule

In [ ]:
for k in (1, 2, 3):
    share = stats.norm.cdf(k) - stats.norm.cdf(-k)
    print(f"within {k} SD of the mean: {share:.4f}")

That is the 68–95–99.7 rule, and now you have seen where it comes from rather
than being asked to memorise it.

Does real data obey it?

In [ ]:
survey = m9.load_survey()
stress = survey.tpstress.dropna()
z = (stress - stress.mean()) / stress.std(ddof=1)

for k in (1, 2, 3):
    print(f"within {k} SD: observed {(z.abs() <= k).mean():.4f}, "
          f"normal says {stats.norm.cdf(k) - stats.norm.cdf(-k):.4f}")

Close, but not exact — which is the normal state of affairs. The normal
distribution is a model, not a description of anything real.

---

## Part 3 — Standardising, and z-scores

Fred sits two tests and scores the same mark in both. In one the class average
was low, in the other it was high. Did he do better in one than the other?

Raw marks cannot answer that. Standardised ones can.

In [ ]:
fred = 78
for name, mean, sd in [("Test A", 70, 8), ("Test B", 66, 6)]:
    print(f"{name}: z = {(fred - mean) / sd:.2f}")

A **z-score** expresses a value as the number of standard deviations it sits
from the mean:

$$z = \frac{x - \bar{x}}{s}$$

The set of z-scores has mean zero and standard deviation one, whatever the
original units were. That is what makes two different scales comparable.

In [ ]:
z = (stress - stress.mean()) / stress.std(ddof=1)
print(f"mean of z = {z.mean():.6f}, SD of z = {z.std(ddof=1):.6f}")

### Tail probabilities without a table

Textbooks send you to a printed table of areas under the normal curve. You have
the function itself.

In [ ]:
print(f"P(Z > 2.00) = {stats.norm.sf(2.00):.4f}")
print(f"P(Z > 2.33) = {stats.norm.sf(2.33):.4f}")
print(f"P(|Z| > 1.96) = {2 * stats.norm.sf(1.96):.4f}")

A worked question of the kind the tables were for. If 480 scores are normally
distributed with a mean of 60 and a standard deviation of 8, how many are 76 or
above?

In [ ]:
z_76 = (76 - 60) / 8
share = stats.norm.sf(z_76)
print(f"z = {z_76:.2f}, tail area = {share:.4f}, expected count = {share * 480:.1f}")

And the head-injury example: someone with a head injury scores 89 on a
comprehension test where uninjured people average 92 with a standard deviation
of 6. Is that impaired?

In [ ]:
z_89 = (89 - 92) / 6
print(f"z = {z_89:.2f}")
print(f"share of uninjured people scoring this low or lower: {stats.norm.cdf(z_89):.4f}")

Nearly a third of uninjured people would score at least this low. On this
evidence alone, no.

**Note what just happened.** We compared an observation to a distribution and
asked how surprising it was. That is the whole logic of the tests in weeks 3 to
5, and you have now done it once by hand.

---

## Part 4 — Shape

Centre and spread are not enough. Two variables can share both and look nothing
alike.

In [ ]:
m9.describe(festival, ["day1", "day2", "day3"])

**Skew** is asymmetry. Positive means a tail to the right, negative a tail to
the left, zero means symmetric.

**Kurtosis** is about the tails: how much of the variance comes from rare
extreme values rather than typical ones. Both `scipy` and this module report
*excess* kurtosis, so zero means "like a normal".

In [ ]:
for name, sample in [
    ("normal", rng.normal(0, 1, 20_000)),
    ("right-skewed", rng.gamma(1.6, 1.0, 20_000)),
    ("uniform (light tails)", rng.uniform(-2, 2, 20_000)),
]:
    print(f"{name:24s} skew {stats.skew(sample):+.2f}  kurtosis {stats.kurtosis(sample):+.2f}")

**Treat both as descriptions of shape.** They are not tests, and we will not use
them as pass/fail criteria. Week 3 explains why that matters at this module's
sample sizes.

### Outliers

In [ ]:
q1, q3 = day1.quantile([0.25, 0.75])
iqr = q3 - q1
fence = q3 + 1.5 * iqr

flagged = day1[day1 > fence]
print(f"upper fence = {fence:.2f}")
print(f"values beyond it: {len(flagged)}, largest = {flagged.max():.2f}")

The hygiene scale runs from 0 to 4. A value of twenty is not an unusually clean
festival-goer; it is a data-entry error. Compare:

In [ ]:
clean = m9.load_festival(with_outlier=False).day1.dropna()

comparison = pd.DataFrame({
    "as recorded": [day1.mean(), day1.std(ddof=1), day1.median(),
                    day1.quantile(.75) - day1.quantile(.25)],
    "corrected": [clean.mean(), clean.std(ddof=1), clean.median(),
                  clean.quantile(.75) - clean.quantile(.25)],
}, index=["mean", "SD", "median", "IQR"])
comparison.round(3)

One value out of eight hundred moves the standard deviation substantially and
leaves the median and IQR untouched. That is the difference between a
**resistant** summary and one that is not.

**A value beyond the fence is not a licence to delete it.** It is an instruction
to find out what it is. This one is an error and gets corrected. A genuine
extreme value stays, and gets reported.

---

## Part 5 — Choosing a picture

The rule is short:

| Variable | Plot |
|---|---|
| One continuous variable | histogram, or a density plot |
| One continuous variable, compared across groups | boxplots side by side |
| One categorical variable | bar chart, or just a table |
| Two continuous variables | scatterplot (week 3) |

### Histograms, and the bin width

A histogram is not a fact about the data. It is a fact about the data *and* your
choice of bin width.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, width in zip(axes, (1.0, 0.4, 0.05)):
    ax.hist(clean, bins=np.arange(clean.min(), clean.max() + width, width))
    ax.set_title(f"bin width {width}")
    ax.set_xlabel("hygiene, day 1")
fig.tight_layout();

Too wide and you lose the shape; too narrow and you are looking at noise. Try
more than one before you believe any of them.

`m9.histogram_with_normal` draws the histogram on the **density** scale with a
matching normal curve over it, so the two are comparable:

In [ ]:
m9.histogram_with_normal(clean, xlabel="Hygiene, day 1 (corrected)");

### Density plots

A density plot is a smoothed histogram. The smoothing has a parameter, and it
makes the same kind of difference the bin width does:

In [ ]:
grid = np.linspace(clean.min() - 0.5, clean.max() + 0.5, 400)

for bw in (0.08, 0.3, 1.0):
    plt.plot(grid, stats.gaussian_kde(clean, bw_method=bw)(grid), label=f"bandwidth {bw}")
plt.xlabel("hygiene, day 1")
plt.ylabel("density")
plt.legend();

### Boxplots

A boxplot is the five-number summary drawn: minimum, Q1, median, Q3, maximum,
with anything beyond the fences plotted individually.

In [ ]:
m9.grouped_box(festival, "day1", "location", ylabel="Hygiene, day 1");

Read it in this order: where the median sits inside the box (off-centre means
skew), how tall the box is (that is the IQR), how long the whiskers are, and
what is beyond them.

### Bar charts, and one to avoid

For a categorical variable, a bar chart of counts:

In [ ]:
romcom = m9.load_romcom()
m9.frequency(romcom, "film")

You will also see bar charts of **group means**. Avoid them.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)

romcom.groupby("film").interest.mean().plot.bar(ax=axes[0], rot=0)
axes[0].set_title("Two numbers")

jitter = np.random.default_rng(1)
films = romcom.groupby("film").interest
for i, (film, values) in enumerate(films):
    axes[1].scatter(i + jitter.uniform(-0.08, 0.08, len(values)), values, alpha=0.7)
axes[1].set_xticks(range(films.ngroups), [film for film, _ in films])
axes[1].set_title("Forty observations")
fig.tight_layout();

Both panels show the same data. The left one shows two numbers and hides the
spread, the group sizes and the overlap. The right one, a dot plot, shows every
observation — and so all three, for about the same amount of ink.

---

## Part 6 — Describing a whole dataset

Everything above is per variable. Now the report your reader actually needs.

In [ ]:
survey = m9.load_survey()
survey.shape

The Melbourne wellbeing survey: responses from members of the general public,
measured on validated scales for self-esteem, optimism, perceived control,
perceived stress, positive and negative affect, and life satisfaction, plus a
social-desirability scale and demographics.

### Step 1: what question are you answering?

> Does optimism predict life satisfaction, once perceived stress and self-esteem
> are taken into account?

That question names four variables. Describe **those**, together with the
demographics that say who the respondents were — not all 134 columns.

In [ ]:
model_vars = ["toptim", "tpstress", "tlifesat", "tslfest"]

### Step 2: type every variable you will use

In [ ]:
m9.md("""
| Variable | Concept | Level of measurement |
|---|---|---|
| `sex` | Gender | nominal |
| `age` | Age in years | ratio |
| `child` | Has children | nominal |
| `educ` | Highest level of education completed | **ordinal** |
| `smoke` | Smoker | nominal |
| `toptim` | Optimism, six-item Life Orientation Test total | interval, treated as scale |
| `tpstress` | Perceived Stress Scale total, ten items | interval, treated as scale |
| `tlifesat` | Satisfaction With Life total, five items | interval, treated as scale |
| `tslfest` | Rosenberg Self-Esteem total, ten items | interval, treated as scale |
""")

Two of those deserve a note.

**`educ` is ordinal**, not nominal. The categories are ranked. Software will not
know that: the labels are stored as text, so every table sorts them
alphabetically. Step 3 shows it, and fixes it.

**`age` is ratio as recorded here** — whole years, with a true zero. The same
concept collected in bands (18–29, 30–44, 45 and over) would be ordinal. The
level belongs to the measurement, not to the concept.

**The scale totals are sums of ordinal Likert items.** Strictly, adding ordinal
items does not produce an interval measure. In practice, validated psychometric
scales are treated as continuous, and the whole literature does so. That is a
defensible convention rather than a mathematical fact, and you should be able to
say which it is.

### Step 3: describe the categorical variables

In [ ]:
for col in ["sex", "child", "smoke"]:
    display(m9.frequency(survey, col))

In [ ]:
m9.frequency(survey, "educ")

Alphabetical, because the labels are text — so "primary" sits between
"postgraduate" and "some additional training". The real order comes from the
questionnaire, and you have to impose it:

In [ ]:
education_order = ["PRIMARY", "SOME SECONDARY", "COMPLETED HIGHSCHOOL",
                   "SOME ADDITIONAL TRAINING", "COMPLETED UNDERGRADUATE",
                   "POSTGRADUATE COMPLETED"]
survey["educ"] = pd.Categorical(survey.educ, categories=education_order, ordered=True)
m9.frequency(survey, "educ")

### Step 4: describe the continuous variables

In [ ]:
m9.describe(survey, ["age"] + model_vars)

### Step 5: say what is missing

In [ ]:
missing = m9.missingness(survey, model_vars)
print(missing.attrs["summary"])
missing

### Step 6: write it

A description your reader can act on covers:

1. **Who** the participants were, and how they were identified.
2. **How** they were sampled — invited, self-selected, incentivised? Was this
   secondary data, and if so what is known about its collection?
3. **What** was measured, with what instrument, and what is known about that
   instrument's validity and reliability.
4. **Each variable** you will use: its level of measurement, how much of it you
   have, and its summary statistics.
5. **What is missing**, and what that costs.

In [ ]:
m9.md(f"""
This analysis uses data condensed by Julie Pallant from a study of the factors
affecting psychological adjustment and wellbeing. A survey containing validated
scales for self-esteem, optimism, perceived control, perceived stress, positive
and negative affect and life satisfaction was distributed to members of the
general public in Melbourne, Australia and surrounding districts. Participation
was voluntary, so the sample is self-selected and cannot be assumed
representative of the Melbourne population.

The dataset contains {len(survey)} responses on {survey.shape[1]} variables.
Respondents' ages range from {survey.age.min():.0f} to {survey.age.max():.0f}
(median {survey.age.median():.0f}). The four variables used in this analysis are
complete for {survey[model_vars].dropna().shape[0]} respondents; listwise
deletion across them loses
{len(survey) - survey[model_vars].dropna().shape[0]} cases.
""")

Every number in that paragraph was generated. Change the data, run it again,
and the paragraph changes with it. **This is the standard for every description
you write.**

---

## What to take from this week

- A statistic varies from sample to sample. The **standard error** measures how
  much, and it shrinks with the square root of the sample size.
- The **Central Limit Theorem** is about the sampling distribution of the mean,
  not about your data, and "large enough" depends on the population.
- **Standardising** puts different scales on one ruler and turns a value into a
  probability.
- Describe **centre, spread and shape** — and look at the shape before choosing
  the centre.
- A histogram's bin width and a density plot's bandwidth are **your choices**.
  Try several.
- An outlier is something to investigate, not something to delete.
- Describe the variables your question needs, not every column you were given.

## Exercise

`exercise.md` gives you a variable with a known problem in it and asks you to
describe it properly.